In [84]:
from fredapi import Fred
import pandas as pd
import numpy as np
import yfinance as yf
from pandas_datareader.data import DataReader


API_KEY = '60c97d27ab6c225f88849013d178b0d9'


# Grabbing data with FRED API
Here I download 13 raw datapoints which we can use for the JDKF

I didn't know what these were, writing down for me: 
- Core CPI is CPI minus food and fuel prices, because these can move around a lot due to wars etc.
- PPI is the producer price index which is what producers are paying for their supplies
- BAA yield is the yield of corperate bonds from companies rated 'Baa' by Moody's

In [118]:
fred = Fred(api_key=API_KEY)

series_codes = {
    "real_gdp": "GDPC1",
    "industrial_production": "INDPRO",
    "unemployment_rate": "UNRATE",
    "nonfarm_payrolls": "PAYEMS",
    "cpi": "CPIAUCSL",
    "core_cpi": "CPILFESL",
    "ppi": "PPIACO",
    "fed_funds_rate": "FEDFUNDS",
    "baa_yield": "BAA",
    "ten_year_yield": "GS10",
    "one_year_yield": "GS1",
    "house_price_index": "QUSR628BIS",
    "oil_price": "WTISPLC",
}

df = pd.DataFrame({
    name: fred.get_series(code)
    for name, code in series_codes.items()
})

df = df.loc["1970-01-01":"2025-12-31"]

I am deciding wheter we want to create quarterly data by averaging or just taking the last value of the quarter. I think that if we want to capture shocks then having the last datapoint of the quarter would be better. I think both sides can be justified for a lot of these factors, maybe we take an average and a current?

In [ ]:
mean_cols = [
    "real_gdp",
    "industrial_production",
    "unemployment_rate",
    "nonfarm_payrolls",
    "cpi",
    "core_cpi",
    "ppi",
    "fed_funds_rate",
    "baa_yield",
    "ten_year_yield",
    "one_year_yield"

]

last_cols = [
    "house_price_index",
    "oil_price"
]

quarterly_mean = df[mean_cols].resample("QE").mean()
quarterly_last = df[last_cols].resample("QE").last()

quarterly = pd.concat(
    [quarterly_mean, quarterly_last],
    axis=1
)

FRED only had daily sp500 data going back to 2016, I use yfinance to get sp500 data going back to 1970

In [ ]:
sp_raw = yf.download(
    "^GSPC",
    start="1970-01-01",
    auto_adjust=True,
    progress=False
)

sp = sp_raw["Close"]

# Convert from one-column DataFrame to Series if needed
if isinstance(sp, pd.DataFrame):
    sp = sp.squeeze()

sp.name = "sp500"
sp.index = pd.to_datetime(sp.index)

sp_q = sp.resample("QE").last()

quarterly = quarterly.join(sp_q, how="left")

In [117]:
quarterly.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 224 entries, 1970-03-31 to 2025-12-31
Freq: QE-DEC
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   real_gdp               224 non-null    float64
 1   industrial_production  224 non-null    float64
 2   unemployment_rate      224 non-null    float64
 3   nonfarm_payrolls       224 non-null    float64
 4   cpi                    224 non-null    float64
 5   core_cpi               224 non-null    float64
 6   ppi                    224 non-null    float64
 7   fed_funds_rate         224 non-null    float64
 8   baa_yield              224 non-null    float64
 9   ten_year_yield         224 non-null    float64
 10  one_year_yield         224 non-null    float64
 11  oil_price              224 non-null    float64
 12  house_price_index      224 non-null    float64
 13  sp500                  224 non-null    float64
dtypes: float64(14)
memory usag

In [157]:
quarterly.head()

,real_gdp,industrial_production,unemployment_rate,nonfarm_payrolls,cpi,core_cpi,ppi,fed_funds_rate,baa_yield,ten_year_yield,one_year_yield,oil_price,house_price_index,sp500
1970-03-31,5300.652,37.961700,4.166667,71310.666667,38.100000,39.833333,36.633333,8.573333,8.756667,7.366667,7.553333,3.350000,60.6714,89.629997
1970-06-30,5308.164,37.752867,4.766667,71167.000000,38.633333,40.566667,36.833333,7.886667,8.976667,7.713333,7.453333,3.350000,60.1158,72.720001
1970-09-30,5357.077,37.617667,5.166667,70978.000000,39.033333,41.100000,37.033333,6.706667,9.410000,7.460000,6.936667,3.310000,60.3841,84.300003
1970-12-31,5299.672,36.804467,5.833333,70574.000000,39.600000,41.766667,37.100000,5.566667,9.276667,6.853333,5.646667,3.393333,60.3287,92.150002
1971-03-31,5443.619,37.514033,5.933333,70844.000000,39.933333,42.166667,37.600000,3.856667,8.530000,6.016667,4.050000,3.560000,61.5101,100.309998


### Number of NA values for each datapoint (before turning into quarterly data)


In [179]:
import pandas as pd

# 1. Initialize a list to hold the row data
summary_data = []

# 2. Helper function to calculate the longest flatline period
def get_max_flatline(series):
    # If the series has fewer than 2 elements, flatline is 0
    if len(series) < 2:
        return 0
    return (series.diff() == 0).astype(int).groupby((series.diff() != 0).cumsum()).sum().max()

# 3. Gather metrics for each column in df
for col in df.columns:
    summary_data.append({
        "Column Name": col,
        "N Null": df[col].isna().sum(),
        "N Not-Null": df[col].notnull().sum(),
        "N Unique": df[col].nunique(),
        "Max Flatline": get_max_flatline(df[col])
    })

# 4. Gather metrics for the standalone sp series
summary_data.append({
    "Column Name": "sp500",
    "N Null": sp.isna().sum(),
    "N Not-Null": sp.notnull().sum(),
    "N Unique": sp.nunique(),
    "Max Flatline": get_max_flatline(sp)
})

# 5. Convert to a DataFrame and display without the row index numbers
summary_df = pd.DataFrame(summary_data)
summary_df


,Column Name,N Null,N Not-Null,N Unique,Max Flatline
0,real_gdp,448,224,224,0
1,industrial_production,0,672,671,0
2,unemployment_rate,1,671,74,4
3,nonfarm_payrolls,0,672,666,1
4,cpi,1,671,643,2
5,core_cpi,1,671,660,2
6,ppi,0,672,524,3
7,fed_funds_rate,0,672,409,12
8,baa_yield,0,672,475,1
9,ten_year_yield,0,672,482,1


- **N Null**: 448 missing datapoints for quarterly data is expected. There is however one missing datapoint for `unemployement_rate`, `cpi`, and `core_cpi`. I imagine that we can linearly interpolate for this small discrepency, or find new data. I think that it is justifiable to interpolate here.
- **N Not-Null vs. N Unique**: The repeats seem to be due to rounding. The `unumployment_rate` looked suspicious, but the values are provided to one decimal-place and the mean is 6.1 with a standard deviation of 1.7.
- **Max Flatline** (longest stretch of the same value): I am suspicious of `unemployment_rate`, `fed_funds_rate`, and `oil_price`.
    - `unemployment_rate`: At the height of the dot-com bubble, US unemployment remained steadily low (1. I suspect they are using the same data as FRED anyway, but google seems to agree that this was a steady time)
    - `fed_funds_rate`: Fed funds rate remained unchanged from August 2023-2024 so this looks fine (2)
    - `oil_price`:  In the 70s oil prices were set by the oil companies called "posted prices", rather than being set by the free-market (3), so the prices were extremely stable. The spot price of oil only didn't exist on the free market until 1983. The posted prices were not what was actually charged, so this seems like a bit of a poor indicator. There is a monthly consumer price index for fuel oil and other fuels in the US which goes back to 1970 (4), but I am not sure if this is a replacement for pure oil price.

1. https://www.macrotrends.net/1316/us-national-unemployment-rate
2. https://www.gam.com/en/our-thinking/multi-asset-blog/fed-leaves-interest-rates-unchanged-as-expected-august-2024
3. https://blogs.worldbank.org/en/developmenttalk/how-do-current-oil-market-conditions-differ-those-during-price-shocks-1970s
4. https://fred.stlouisfed.org/series/CUSR0000SEHE

### Feature engineering